# SupportLens: train and evaluate Layer 1 on Colab

This notebook trains the two learned Layer 1 models on the annotated training split and then runs the
evaluation script in strict mode. Nothing is simulated: every number written to `data/evaluation/` comes from
the evaluation script run in this session.

1. Qwen2.5-7B-Instruct with a LoRA adapter (4-bit NF4 base, fits a Colab T4).
2. RoBERTa-base token classifier (baseline 3).
3. `python scripts/run_evaluation.py --strict` (zero-shot Qwen, fine-tuned Qwen, RoBERTa, regex baseline).

**Runtime:** Runtime > Change runtime type > T4 GPU.
**Input:** a zip of the project folder that contains `data/ground_truth/train.jsonl`, `val.jsonl` and `test.jsonl`.

## 1. Check the runtime

In [ ]:
import subprocess, sys

print(sys.version)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "no nvidia-smi output")
import torch

assert torch.cuda.is_available(), "No GPU: choose Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

## 2. Get the project

Set `PROJECT_SOURCE` to `"drive"` (zip stored in Google Drive at `DRIVE_ZIP`) or `"upload"` (file picker).

In [ ]:
import os, shutil, zipfile

PROJECT_SOURCE = "upload"  # "drive" or "upload"
DRIVE_ZIP = "/content/drive/MyDrive/supportlens.zip"
WORKDIR = "/content/supportlens"

if PROJECT_SOURCE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    zip_path = DRIVE_ZIP
else:
    from google.colab import files

    uploaded = files.upload()
    zip_path = next(iter(uploaded))

shutil.rmtree(WORKDIR, ignore_errors=True)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(WORKDIR)

entries = os.listdir(WORKDIR)
if len(entries) == 1 and os.path.isdir(os.path.join(WORKDIR, entries[0])):
    WORKDIR = os.path.join(WORKDIR, entries[0])  # the zip contains one top-level folder
os.chdir(WORKDIR)
print(os.getcwd(), sorted(os.listdir(".")))

## 3. Install the dependencies

In [ ]:
!pip install -q -r requirements.txt -r requirements-gpu.txt

## 4. Check the annotated splits

In [ ]:
from pathlib import Path

for name in ("train", "val", "test"):
    path = Path("data/ground_truth") / f"{name}.jsonl"
    assert path.exists(), f"{path} is missing: add the annotated splits to the project zip"
    with open(path, encoding="utf-8") as handle:
        print(name, sum(1 for line in handle if line.strip()), "records")

## 5. Train Qwen2.5-7B-Instruct with LoRA

LoRA on `q_proj`, `k_proj`, `v_proj`, `o_proj` (r=128, alpha=256, dropout 0.05), learning rate 2e-4, 3 epochs,
effective batch size 8, seed 42, 4-bit NF4 base. The best epoch by validation loss is kept.
r=128 trains about 80.7 M parameters, about 1.06 % of the 7.62 B base (thesis 3.3.2: about 1-2 %);
the run records the measured percentage and warns when it falls outside that range.
The adapter, `training_log.json` and `run_metadata.json` are written to `models/qwen_lora_adapters/`.

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!python -m src.layer1_ai_extraction.train_qwen_lora --max-seq-len 2048

In [ ]:
import json

with open("models/qwen_lora_adapters/run_metadata.json", encoding="utf-8") as handle:
    meta = json.load(handle)
print("GPU:", meta["environment"]["hardware"]["gpu_name"], "| CUDA:", meta["environment"]["hardware"]["cuda_version"])
print("trainable parameters:", meta["trainable_parameters"], f"({meta['trainable_percent']:.4f} %)")
print("within the thesis range of 1-2 %:", meta["trainable_percent_within_thesis_range"])
with open("models/qwen_lora_adapters/training_log.json", encoding="utf-8") as handle:
    for epoch in json.load(handle)["epochs"]:
        print(epoch)

## 6. Train the RoBERTa-base baseline

Full fine-tuning, AdamW, learning rate 2e-5, batch size 16, 5 epochs, seed 42, class-weighted cross-entropy
(rare VERSION and TEMPORAL labels get a higher weight). The model is written to `models/roberta_baseline/`.
At the end the log prints `val B-VERSION: f1=...`; an F1 of 0 or `predicted=0` means the model still never
predicts versions, so stop and report it before running the evaluation.

In [ ]:
!python -m src.layer1_ai_extraction.train_roberta_baseline

## 7. Evaluate (strict)

`--strict` makes every model extractor raise instead of falling back to another method, so a missing adapter or
model stops the run.

In [ ]:
!python scripts/run_evaluation.py --strict

## 8. Download the results

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

RESULT_DIRS = ["data/evaluation", "models/qwen_lora_adapters", "models/roberta_baseline"]
bundle = "/content/supportlens_results.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for directory in RESULT_DIRS:
        for path in sorted(Path(directory).rglob("*")):
            if path.is_file():
                archive.write(path, path.as_posix())
print(Path(bundle).stat().st_size / 1e6, "MB")
files.download(bundle)